[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C13_RL_Foundations_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与 RL 方法论热身

本课全程 **纯 numpy、CPU 可跑**，环境是手写的 **GridWorld / toy MDP**，小到能枚举全部状态、能解析求真值。

这个 notebook 做三件事：① 确认环境；② 用最小例子体会 RL 与监督学习的根本不同（**回报是折扣累积、奖励延迟**）；③ 立下全课的两条纪律——**对拍真值** 与 **固定种子复现**。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于画 return/regret 曲线）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 回报：RL 最大化的不是单步奖励，而是折扣累积回报

回报 $G_t=\sum_{k\ge0}\gamma^k r_{t+k+1}$。折扣因子 $\gamma$ 既保证无限和收敛（几何级数），又编码「多看重未来」。

先把这个最基本的量算清楚：给定一串奖励，从每个时刻往后算折扣回报。

In [ ]:
def discounted_returns(rewards, gamma):
    '''给定一回合的奖励序列 rewards[0..T-1]，返回每个时刻的折扣回报 G_t（反向累积）。'''
    G = np.zeros(len(rewards), dtype=float)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running      # G_t = r_t + γ G_{t+1}
        G[t] = running
    return G

rewards = [0.0, 0.0, 0.0, 1.0]      # 奖励只在最后一步出现（延迟回报！）
G = discounted_returns(rewards, gamma=0.9)
print('奖励序列  :', rewards)
print('折扣回报 G:', np.round(G, 4))
# 最后一步回报=1；倒数第二步=0.9；再前一步=0.81 ... 即 γ^(距离)
assert np.allclose(G, [0.9**3, 0.9**2, 0.9**1, 0.9**0]), '延迟奖励按 γ^距离 回传'
print('✅ 唯一的奖励(终点+1)被折扣回传到每个时刻 —— 这就是「信用分配」最朴素的样子')

**关键结论**：那个终点的 `+1` 奖励，被 $\gamma$ 的幂次「回传」给了之前每一步——越远的步数折扣越多。

**这正是信用分配（credit assignment）的最简形式**：延迟到来的回报，按距离加权地归因给此前的动作。后面的 TD、GAE 都是更精巧的回传方式。

## 3 · 有效视界：γ 决定 agent「看多远」

几何级数 $\sum_{k\ge0}\gamma^k=\frac{1}{1-\gamma}$ 给出「有效视界」约 $1/(1-\gamma)$ 步。$\gamma$ 越接近 1 越有远见。

In [ ]:
def effective_horizon(gamma):
    return 1.0 / (1.0 - gamma)

for g in [0.0, 0.5, 0.9, 0.99]:
    print(f'γ={g:<5} -> 有效视界 ≈ {effective_horizon(g):>6.1f} 步,  Σγ^k = {effective_horizon(g):.1f}')
# γ=0 完全短视(只看即时奖励)；γ=0.99 看~100 步
assert abs(effective_horizon(0.9) - 10.0) < 1e-9
assert abs(effective_horizon(0.99) - 100.0) < 1e-9
# 验证几何级数：截断求和应逼近 1/(1-γ)
g = 0.9
approx = sum(g**k for k in range(200))
assert abs(approx - effective_horizon(g)) < 1e-6
print('✅ 有效视界 ≈ 1/(1-γ)：γ 是 agent 远见程度的旋钮')

## 4 · 第一条纪律：对拍真值

本课每个无模型算法都要和一个**解析/精确的 ground truth** 比对。先体会最简单的情形：一个 2 状态的链，用解析解和迭代解两种方式求同一个量，对拍它们一致。

考虑只有奖励、无动作的「马尔可夫奖励过程」：状态 0 每步得 `r0` 后转到状态 1，状态 1 是吸收态（自环、奖励 0）。状态 0 的价值 $V(0)=r_0+\gamma V(1)$，$V(1)=0+\gamma V(1)\Rightarrow V(1)=0$。

In [ ]:
gamma = 0.9
r0 = 2.0
# 解析解：V(1)=0, V(0)=r0 + γ*0 = r0
V_analytic = np.array([r0, 0.0])

# 迭代解（价值迭代的雏形）：反复用 Bellman 备份直到收敛
V = np.zeros(2)
P = np.array([[0.0, 1.0], [0.0, 1.0]])    # 0->1, 1->1(吸收)
R = np.array([r0, 0.0])
for _ in range(200):
    V = R + gamma * (P @ V)

print('解析解 V:', V_analytic)
print('迭代解 V:', np.round(V, 6))
assert np.allclose(V, V_analytic, atol=1e-6), '迭代解必须对拍解析解'
print('✅ 对拍通过：迭代 Bellman 备份收敛到解析精确解 —— 这是全课验证算法的范式')

## 5 · 第二条纪律：固定种子，结果可复现

RL 实验随机性极大。固定 `np.random.default_rng(seed)` 保证重跑得到**逐位相同**的结果。

下面用同一个种子跑两次随机 rollout，验证它们完全一致；再用不同种子，验证结果不同——这就是为什么报告 RL 结果必须说明种子。

In [ ]:
def random_rollout(n_steps, seed):
    rng = np.random.default_rng(seed)
    return rng.integers(0, 4, size=n_steps)        # 模拟随机选动作(4 个动作)

run_a = random_rollout(10, seed=0)
run_b = random_rollout(10, seed=0)
run_c = random_rollout(10, seed=1)
print('seed=0 第一次:', run_a)
print('seed=0 第二次:', run_b)
print('seed=1       :', run_c)
assert np.array_equal(run_a, run_b), '同种子必须逐位相同（可复现）'
assert not np.array_equal(run_a, run_c), '不同种子结果应不同'
print('✅ 同种子可复现、异种子有差异 —— 严肃 RL 实验必须固定并报告种子')

## 6 · 一个会贯穿全课的 GridWorld 预览

本课主力环境是 GridWorld：agent 在网格上走，到目标 +1、掉陷阱 -1、每步小惩罚。下面建一个极小的 3×3 版，只跑通「reset → step → done」的接口（完整版在模块 01）。这就是后面所有算法的「驾校训练场」。

In [ ]:
class TinyGridWorld:
    '''3x3 网格：起点(0,0)，目标(2,2)+1，每步-0.04。动作 0/1/2/3 = 上/右/下/左。'''
    def __init__(self):
        self.n_rows, self.n_cols = 3, 3
        self.goal = (2, 2)
        self.moves = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}
    def reset(self):
        self.pos = (0, 0)
        return self.pos
    def step(self, a):
        dr, dc = self.moves[a]
        r, c = self.pos
        nr, nc = min(max(r + dr, 0), self.n_rows - 1), min(max(c + dc, 0), self.n_cols - 1)
        self.pos = (nr, nc)
        done = (self.pos == self.goal)
        reward = 1.0 if done else -0.04
        return self.pos, reward, done

env = TinyGridWorld()
s = env.reset()
# 手动走一条最优路径：右右下下 -> 到目标
path = [1, 1, 2, 2]
total = 0.0
for a in path:
    s, r, done = env.step(a)
    total += r
print('终点状态:', s, '| 累积奖励:', round(total, 3), '| done:', done)
assert s == (2, 2) and done, '应当走到目标'
assert abs(total - (1.0 - 3 * 0.04)) < 1e-9, '4 步：3 步各 -0.04，最后一步 +1'
print('✅ GridWorld 接口跑通：reset/step/done 正常，奖励结构正确')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的两条契约**：① 你写的每个算法都会**对拍解析/DP 真值**（VI/PI 收敛同一 $V^*$、Q-learning 学到 DP 的 $\pi^*$）；② 每个随机实验都**固定种子可复现**。结构正确则对拍通过，对拍通过则逻辑可迁移到深度 RL。

**接下来六个模块**：01 MDP & Bellman → 02 TD & Q-learning → 03 策略梯度 → 04 Actor-Critic & GAE → 05 老虎机与探索。每一步都建立在前一步之上。

下一站：**模块 01 · MDP 与 Bellman 方程**。